# Chameleon setup notebook for Smart Transaction Categorization
Use this notebook inside the Chameleon Jupyter environment. It creates a lease, a VM, the required security groups, and a floating IP for your CPU baseline and model-optimization work.

Before you run this notebook, delete any old lease/server with the same name in Horizon. The online evaluation lab uses a `KVM@TACC` `m1.medium` VM and attaches security groups before exposing FastAPI, Jupyter, Prometheus, and Grafana. Your project proposal also expects a single Chameleon VM to be sufficient for the lightweight CPU classifier. fileciteturn12file5 fileciteturn12file13

In [ ]:
from chi import server, context, lease, network
import chi, os, time, datetime, json

# Replace this with your real project suffix if needed.
PROJECT_SUFFIX = "proj08"

context.version = "1.0"
context.choose_project()
context.choose_site(default="KVM@TACC")

username = os.getenv("USER")
lease_name = f"lease-tx-serving-{username}-{PROJECT_SUFFIX}"
server_name = f"node-tx-serving-{username}-{PROJECT_SUFFIX}"
print({"lease_name": lease_name, "server_name": server_name})


## 1. Create a 6-hour lease for one `m1.medium` VM

In [ ]:
l = lease.Lease(lease_name, duration=datetime.timedelta(hours=6))
l.add_flavor_reservation(id=chi.server.get_flavor_id("m1.medium"), amount=1)
l.submit(idempotent=True)
l.show()


## 2. Launch the VM with `CC-Ubuntu24.04`

In [ ]:
s = server.Server(
    server_name,
    image_name="CC-Ubuntu24.04",
    flavor_name=l.get_reserved_flavors()[0].name
)
s.submit(idempotent=True)
s.refresh()
s.show()


## 3. Create and attach security groups

In [ ]:
# Helper to create a security group and one ingress TCP rule if needed.
def ensure_tcp_group(name, port):
    groups = network.list_security_groups()
    existing = [g for g in groups if g["name"] == name]
    if not existing:
        sg = network.create_security_group(name=name)
        network.create_security_group_rule(
            security_group_id=sg["id"],
            direction="ingress",
            protocol="tcp",
            port_range_min=port,
            port_range_max=port,
            remote_ip_prefix="0.0.0.0/0",
        )
    return name

sg_names = [
    ensure_tcp_group(f"allow-ssh-{PROJECT_SUFFIX}", 22),
    ensure_tcp_group(f"allow-8000-{PROJECT_SUFFIX}", 8000),
    ensure_tcp_group(f"allow-8888-{PROJECT_SUFFIX}", 8888),
    ensure_tcp_group(f"allow-3000-{PROJECT_SUFFIX}", 3000),
    ensure_tcp_group(f"allow-9090-{PROJECT_SUFFIX}", 9090),
]

for sg in sg_names:
    try:
        s.associate_security_group(sg)
    except Exception:
        pass

print("Attached security groups:", sg_names)


## 4. Allocate and associate a floating IP

In [ ]:
fip = network.get_or_create_floating_ip()
s.associate_floating_ip(fip["floating_ip_address"])
time.sleep(5)
s.refresh()
s.show()
print("Floating IP:", fip["floating_ip_address"])


## 5. Print the SSH command you will run from your Mac

In [ ]:
floating_ip = fip["floating_ip_address"]
print(f"ssh -i ~/.ssh/id_rsa_chameleon cc@{floating_ip}")
print(f"FastAPI docs: http://{floating_ip}:8000/docs")
print(f"Jupyter:      http://{floating_ip}:8888")
print(f"Prometheus:   http://{floating_ip}:9090")
print(f"Grafana:      http://{floating_ip}:3000")


## 6. Commands to run after SSH
Run the next block on the VM after you log in.

In [ ]:
commands = r'''
sudo apt update
curl -sSL https://get.docker.com/ | sudo sh
sudo groupadd -f docker
sudo usermod -aG docker $USER
newgrp docker

git clone <YOUR_GITHUB_REPO_URL>
cd <YOUR_REPO_DIR>

# Baseline service
cd baseline_service
python3 -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
python scripts/make_dummy_model.py --output models/dummy_pipeline.joblib
python scripts/generate_requests.py --output tests/generated_requests.jsonl --count 500
docker build -t tx-baseline .
docker run --rm -d -p 8000:8000 --name tx-baseline tx-baseline
curl http://localhost:8000/healthz
curl http://localhost:8000/readyz

# Model optimization notebook
cd ../model_optimization
docker build -f Dockerfile.modelopt -t tx-modelopt-cpu .
docker run -d --rm -p 8888:8888 -v $(pwd):/workspace --name jupyter tx-modelopt-cpu
docker exec jupyter jupyter server list
'''
print(commands)
